# 实战ReAct Agent

In [1]:
import os
import re
from llm_client import LLMProxy
from tools import ToolExecutor, search

In [2]:
# REACT_PROMPT_TEMPLATE的定义
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
{tools}

请严格按照以下格式进行回应：

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一：
- `{{tool_name}}[{{tool_input}}]`：调用一个可用工具。
- `Finish[最终答案]`：当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在`Action:`字段后使用 `Finish[最终答案]` 来输出最终答案。


现在，请开始解决以下问题：
Question: {question}
History: {history}
"""


In [3]:
class ReActAgent:
    def __init__(self, llm_client: LLMProxy, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        self.history = []
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"\n--- 第 {current_step} 步 ---")

            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(tools=tools_desc, question=question, history=history_str)

            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            if not response_text:
                print("错误：LLM未能返回有效响应。")
                break

            thought, action = self._parse_output(response_text)
            if thought:
                print(f"🤔 思考: {thought}")
            if not action:
                print("警告：未能解析出有效的Action，流程终止。")
                break
            
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = self._parse_action_input(action)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                self.history.append("Observation: 无效的Action格式，请检查。")
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")
            tool_function = self.tool_executor.getTool(tool_name)
            observation = tool_function(tool_input) if tool_function else f"错误：未找到名为 '{tool_name}' 的工具。"
            
            print(f"👀 观察: {observation}")
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        print("已达到最大步数，流程终止。")
        return None

    def _parse_output(self, text: str):
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        return (match.group(1), match.group(2)) if match else (None, None)

    def _parse_action_input(self, action_text: str):
        match = re.match(r"\w+\[(.*)\]", action_text, re.DOTALL)
        return match.group(1) if match else ""

In [4]:
if __name__ == '__main__':
    llm = LLMProxy()
    tool_executor = ToolExecutor()
    search_desc = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    tool_executor.registerTool("Search", search_desc, search)
    agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)
    question = "华为最新的手机是哪一款？它的主要卖点是什么？"
    agent.run(question)

工具 'Search' 已注册。

--- 第 1 步 ---
🧠 正在调用 qwen-max 模型...
✅ 大语言模型响应成功:
Thought: 我需要查找最新的华为手机型号及其主要卖点。由于这些信息可能随着新产品的发布而变化，因此我将使用Search工具来获取最新资料。
Action: Search[华为 最新 手机 型号 主要 卖点]
🤔 思考: 我需要查找最新的华为手机型号及其主要卖点。由于这些信息可能随着新产品的发布而变化，因此我将使用Search工具来获取最新资料。
🎬 行动: Search[华为 最新 手机 型号 主要 卖点]
🔍 正在执行 [SerpApi] 网页搜索: 华为 最新 手机 型号 主要 卖点
👀 观察: [1] 2026年华为手机哪一款性价比高？华为手机推荐与市场分析
在硬件方面，华为在缺失5G芯片之后，其手机业务线被迫停滞，较友商的新手机而言大失优势。现在华为手机最大的卖点只剩下鸿蒙HarmonyOS系统，以及饱受争议的品牌信仰。 这里推荐 ...

[2] 华为手机- 华为官网
智能手机 ; Mate 系列. 非凡旗舰 · HUAWEI Mate 80 Pro Max. ￥7999 起 ; Pura 系列. 先锋影像. HUAWEI Pura 80 Pro+. ￥7999 起 ; Pocket 系列. 美学新篇. HUAWEI Pocket 2 优享 ...

[3] 旗舰手机产品市场盘点月报
2023年5月旗舰手机存量市场中，华为品牌的旗舰产品领跑，其中华为的Mate50及Mate50 Pro机型位列前二， vivo X90机型挤进TOP3。 2023年5月主要旗舰手机产品存量市场表现. 23.6%.

--- 第 2 步 ---
🧠 正在调用 qwen-max 模型...
✅ 大语言模型响应成功:
Thought: 根据搜索结果，华为最新的手机型号包括Mate 80 Pro Max和HUAWEI Pura 80 Pro+等。但是，这些信息可能不是最新的，因为观察到的数据中还提到了2023年的机型如Mate50系列。考虑到技术更新速度非常快，我需要进一步确认目前（即最近）华为发布的最新款手机是什么，并了解它的主要卖点。此外，提到的鸿蒙HarmonyOS系统作为华为手机的一大特色，也是值得注意的一个方面。
Action: Sea